# 02 - Preprocessing Validation

Validate the QC and preprocessing pipeline outputs.  
Shows PCA batch correction plots, protein value distributions before/after log transform,
and QC summary statistics.  
**This notebook reads processed data only** -- it does not rerun preprocessing.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA

from src.preprocessing.somascan_qc import identify_seq_columns

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

processed_path = os.path.join('..', config['paths']['processed'])
print('Config loaded. Processed data directory:', processed_path)

## Load Processed Data

Load the cleaned ADNI proteomics parquet file produced by the preprocessing pipeline.

In [ ]:
df = pd.read_parquet(os.path.join(processed_path, 'adni_proteomics_clean.parquet'))
seq_cols = identify_seq_columns(df)

print(f'Processed data shape: {df.shape}')
print(f'Number of participants: {df["RID"].nunique()}')
print(f'Number of proteins: {len(seq_cols)}')
print(f'Columns with any NaN: {df[seq_cols].isna().any().sum()}')

## PCA Batch Correction Validation

PCA plots colored by batch (PlateId) and by diagnosis (TRAJECTORY).  
After successful batch correction, batch colors should be intermixed while disease groups remain separated.

In [ ]:
# PCA on processed (batch-corrected) data
protein_matrix = df[seq_cols].dropna()
pca = PCA(n_components=2, random_state=config['random_seed'])
pcs = pca.fit_transform(protein_matrix)

df_pca = df.loc[protein_matrix.index].copy()
df_pca['PC1'] = pcs[:, 0]
df_pca['PC2'] = pcs[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color by batch (PlateId)
if 'PlateId' in df_pca.columns:
    n_plates = df_pca['PlateId'].nunique()
    axes[0].scatter(df_pca['PC1'], df_pca['PC2'], c=df_pca['PlateId'].astype('category').cat.codes,
                    cmap='tab20', alpha=0.4, s=10)
    axes[0].set_title(f'After Batch Correction -- Colored by Plate ({n_plates} plates)')
else:
    axes[0].text(0.5, 0.5, 'PlateId column not found', transform=axes[0].transAxes, ha='center')
    axes[0].set_title('After Batch Correction -- Colored by Plate')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

# Color by trajectory
if 'TRAJECTORY' in df_pca.columns:
    for traj, grp in df_pca.groupby('TRAJECTORY'):
        axes[1].scatter(grp['PC1'], grp['PC2'], alpha=0.4, s=10, label=traj)
    axes[1].legend(fontsize=7, markerscale=2)
axes[1].set_title('After Batch Correction -- Colored by Trajectory')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

plt.tight_layout()
plt.show()

## Protein Value Distributions

Distribution of log2-transformed protein values. After preprocessing, each protein should be approximately normally distributed.

In [ ]:
# Sample 20 random proteins and show their distributions
rng = np.random.RandomState(config['random_seed'])
sample_proteins = rng.choice(seq_cols, size=min(20, len(seq_cols)), replace=False)

fig, axes = plt.subplots(4, 5, figsize=(16, 10))
axes = axes.ravel()

for i, prot in enumerate(sample_proteins):
    vals = df[prot].dropna()
    axes[i].hist(vals, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(prot.replace('seq.', ''), fontsize=8)
    axes[i].tick_params(labelsize=6)

plt.suptitle('Distribution of Log2-Transformed Protein Values (20 random proteins)', fontsize=12)
plt.tight_layout()
plt.show()

## QC Summary Statistics

Summary of samples and proteins removed at each QC step.

In [ ]:
# Overall summary of the processed dataset
print('=== Processed Data Summary ===')
print(f'Total samples (visits): {len(df)}')
print(f'Unique participants:    {df["RID"].nunique()}')
print(f'Proteins after QC:      {len(seq_cols)}')
print(f'Missing values (total): {df[seq_cols].isna().sum().sum()}')
print(f'Missing rate:           {df[seq_cols].isna().mean().mean():.4%}')
print()

# Per-protein summary
protein_stats = pd.DataFrame({
    'mean': df[seq_cols].mean(),
    'std': df[seq_cols].std(),
    'min': df[seq_cols].min(),
    'max': df[seq_cols].max(),
    'missing_frac': df[seq_cols].isna().mean()
})
print('Per-protein statistics (across all samples):')
print(protein_stats.describe().round(3))

# Visits per participant
visits_per_participant = df.groupby('RID')['VISCODE'].nunique()
print(f'\nVisits per participant: mean={visits_per_participant.mean():.1f}, '
      f'median={visits_per_participant.median():.0f}, '
      f'range=[{visits_per_participant.min()}, {visits_per_participant.max()}]')